# Limpieza de Datos 

## 1. carga

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from config.rutas import RUTA_DATA_RAW
df = pl.read_csv(RUTA_DATA_RAW / "endireh_2021.csv")
print(df.shape)
df.head()


## 2. Variables no consideradas todavía

`eda.ipynb` y `medidas_localizacion.ipynb` solo se habian visto  3 variables cuantitativas (`edad_primer_union`, `num_hijos`, `ingreso_pareja`) y 3 cualitativas (`estado_civil_desc`, `estado_civil_id`, `nivel_escolaridad`) en eda. Revisamos el resto antes de dar por terminada la limpieza.

In [9]:
schema = df.schema
print("Todas las columnas del dataset:")
for col, tipo in schema.items():
    print(f" {col}: {tipo}")

Todas las columnas del dataset:
 cve_entidad: Int64
 nom_entidad: String
 cve_municipio: Int64
 nom_municipio: String
 edad_primer_union: Float64
 num_hijos: Float64
 nivel_escolaridad: String
 estado_civil_id: Int64
 estado_civil_desc: String
 estrato_socioeconomico: Int64
 pareja_trabaja_id: Int64
 pareja_trabaja_desc: String
 ingreso_pareja: Float64
 dinero_propio_id: Int64
 dinero_propio_desc: String
 apoyo_gobierno_id: Int64
 apoyo_gobierno_desc: String
 tiene_ahorros_id: Int64
 tiene_ahorros_desc: String
 propietaria_vivienda_id: Int64
 propietaria_vivienda_desc: String
 sufrio_violencia_pareja: Int64
 factor_expansion: Int64
 anio_encuesta: Int64


Columnas que **no se han tocado** en `eda.ipynb` ni en `medidas_localizacion.ipynb`:

| Variable | Por qué revisarla |
|---|---|
| `cve_entidad`, `cve_municipio` | Claves numéricas (id); tenemos que verificar que no tengan nulos y sean consistentes con `nom_entidad`/`nom_municipio` según la especificaión del dataset. |
| `nom_municipio` | Cualitativa nominal con 1,206 valores únicos; revisar nulos e inconsistencias de escritura. |
| `estrato_socioeconomico` | Según el descriptor oficial es una variable de **diseño muestral geográfico**, no un índice de ingreso del hogar. Revisar nulos. |
| `pareja_trabaja_id`, `pareja_trabaja_desc` | Se usan en la sección de imputación de `ingreso_pareja`. |
| `dinero_propio_id/desc`, `apoyo_gobierno_id/desc`, `tiene_ahorros_id/desc`, `propietaria_vivienda_id/desc` | No exploradas aún; binarias, revisar nulos y frecuencias. |
| `sufrio_violencia_pareja` | Dado que se usa como variable de agrupación en medidas de variabilidad tenemos que confirmar que no tiene nulos. |
| `anio_encuesta` | Constante (2021); se descarta como variable de análisis por no tener varianza. |

In [10]:
columnas_no_revisadas = [
    "cve_entidad", "cve_municipio", "nom_municipio",
    "estrato_socioeconomico",
    "pareja_trabaja_id", "pareja_trabaja_desc",
    "dinero_propio_id", "dinero_propio_desc",
    "apoyo_gobierno_id", "apoyo_gobierno_desc",
    "tiene_ahorros_id", "tiene_ahorros_desc",
    "propietaria_vivienda_id", "propietaria_vivienda_desc",
    "sufrio_violencia_pareja",
]

resumen_no_revisadas = df.select([
    pl.col(c).null_count().alias(f"{c}__nulos") for c in columnas_no_revisadas
])
print(resumen_no_revisadas.transpose(include_header=True, header_name="columna", column_names=["nulos"]))

for c in columnas_no_revisadas:
    print(f"\n{c} — valores únicos:")
    print(df[c].value_counts().sort("count", descending=True))

shape: (15, 2)
┌─────────────────────────────────┬───────┐
│ columna                         ┆ nulos │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ cve_entidad__nulos              ┆ 0     │
│ cve_municipio__nulos            ┆ 0     │
│ nom_municipio__nulos            ┆ 0     │
│ estrato_socioeconomico__nulos   ┆ 0     │
│ pareja_trabaja_id__nulos        ┆ 0     │
│ …                               ┆ …     │
│ tiene_ahorros_id__nulos         ┆ 0     │
│ tiene_ahorros_desc__nulos       ┆ 0     │
│ propietaria_vivienda_id__nulos  ┆ 0     │
│ propietaria_vivienda_desc__nul… ┆ 0     │
│ sufrio_violencia_pareja__nulos  ┆ 0     │
└─────────────────────────────────┴───────┘

cve_entidad — valores únicos:
shape: (32, 2)
┌─────────────┬───────┐
│ cve_entidad ┆ count │
│ ---         ┆ ---   │
│ i64         ┆ u32   │
╞═════════════╪═══════╡
│ 29          ┆ 3714  │
│ 20          ┆ 3706  │
│ 11          ┆ 3663  │

## 3. Por qué existen los nulos: relaciones entre variables

### 3.0 Cómo está diseñado el cuestionario

La ENDIREH 2021 aplica una de **3 versiones** de cuestionario a cada mujer de 15+ años:

- **Cuestionario A** — casada o unida actualmente.
- **Cuestionario B** — separada, divorciada o viuda.
- **Cuestionario C** — soltera (nunca unida).

`estado_civil_desc` es la variable del dataset que mejor aproxima esto bifurcación, y por eso es la variable categórica clave para explicar los nulos de variables ligadas a la vida de pareja.

### 3.1 Dos tipos de "nulo" que no deben tratarse igual

- **No aplica (missing estructural):** la celda queda en blanco porque una pregunta filtro determinó que no correspondía. No es un dato perdido — imputar aquí inventaría un evento que no ocurrió, como decir que alguien tiene pareja cuando en realidad no.
- **No sabe / no especificado (missing real):** la pregunta sí aplicaba, pero no se obtuvo respuesta. Codificado con **98/99** (variables de 2 dígitos, ej. edad) y **999998/999999** (variables de 6 dígitos, ej. ingresos). Aquí  donde imputamos
  
  
### 3.2 Relaciones confirmadas entre variables categóricas y numéricas

| Variable numérica | Pregunta filtro oficial | Variable categórica en el dataset | Relación |
|---|---|---|---|
| `edad_primer_union` | P3_8 | `estado_civil_desc` | Solo aplica a mujeres alguna vez unidas; en solteras la pregunta ni se hace. |
| `ingreso_pareja` | P4_3 | `pareja_trabaja_desc` | Filtro directo si P4_3 = 1 (Sí trabaja). |
| `num_hijos` | Sin mapeo 1 a 1 confirmado | `estado_civil_desc` (relación más débil) | No depende estrictamente de tener pareja actual |

In [11]:
total_nulos_edad = df.filter(pl.col("edad_primer_union").is_null()).height
print(f"Total de nulos en edad_primer_union: {total_nulos_edad}")

nulos_edad_por_estado_civil = (
    df.filter(pl.col("edad_primer_union").is_null())
    .group_by("estado_civil_desc")
    .len()
    .sort("len", descending=True)
    .with_columns(
        (pl.col("len") / total_nulos_edad * 100).round(1).alias("% de los nulos")
    )
)
print(nulos_edad_por_estado_civil)

Total de nulos en edad_primer_union: 85567
shape: (6, 3)
┌───────────────────┬───────┬────────────────┐
│ estado_civil_desc ┆ len   ┆ % de los nulos │
│ ---               ┆ ---   ┆ ---            │
│ str               ┆ u32   ┆ f64            │
╞═══════════════════╪═══════╪════════════════╡
│ Casada            ┆ 43560 ┆ 50.9           │
│ Unión libre       ┆ 22443 ┆ 26.2           │
│ Soltera           ┆ 18079 ┆ 21.1           │
│ Separada          ┆ 860   ┆ 1.0            │
│ Divorciada        ┆ 428   ┆ 0.5            │
│ Viuda             ┆ 197   ┆ 0.2            │
└───────────────────┴───────┴────────────────┘


Si la gran mayoría de los nulos de `edad_primer_union` cae en "Soltera" (nunca unida), eso confirma la hipótesis de no-aplicabilidad estructural y refuerza la decisión de no imputar esa variable con un número.

In [12]:
total_nulos_hijos = df.filter(pl.col("num_hijos").is_null()).height
print(f"Total de nulos en num_hijos: {total_nulos_hijos}")

nulos_hijos_por_estado_civil = (
    df.filter(pl.col("num_hijos").is_null())
    .group_by("estado_civil_desc")
    .len()
    .sort("len", descending=True)
    .with_columns(
        (pl.col("len") / total_nulos_hijos * 100).round(1).alias("% de los nulos")
    )
)
print(nulos_hijos_por_estado_civil)

Total de nulos en num_hijos: 19436
shape: (4, 3)
┌───────────────────┬───────┬────────────────┐
│ estado_civil_desc ┆ len   ┆ % de los nulos │
│ ---               ┆ ---   ┆ ---            │
│ str               ┆ u32   ┆ f64            │
╞═══════════════════╪═══════╪════════════════╡
│ Soltera           ┆ 18016 ┆ 92.7           │
│ Separada          ┆ 839   ┆ 4.3            │
│ Divorciada        ┆ 395   ┆ 2.0            │
│ Viuda             ┆ 186   ┆ 1.0            │
└───────────────────┴───────┴────────────────┘


Consulta de contexto: ver si los nulos de `ingreso_pareja` se relacionan con otras variables de situación económica del hogar.

In [13]:
comparacion_economica = (
    df
    .with_columns(
        pl.col("ingreso_pareja").is_null().alias("sin_dato_ingreso_pareja")
    )
    .group_by("sin_dato_ingreso_pareja")
    .agg([
        (pl.col("dinero_propio_desc") == "Sí").mean().alias("% con dinero propio"),
        (pl.col("apoyo_gobierno_desc") == "Sí").mean().alias("% con apoyo de gobierno"),
        (pl.col("tiene_ahorros_desc") == "Sí").mean().alias("% con ahorros"),
    ])
)
print(comparacion_economica)

shape: (2, 4)
┌─────────────────────────┬─────────────────────┬─────────────────────────┬───────────────┐
│ sin_dato_ingreso_pareja ┆ % con dinero propio ┆ % con apoyo de gobierno ┆ % con ahorros │
│ ---                     ┆ ---                 ┆ ---                     ┆ ---           │
│ bool                    ┆ f64                 ┆ f64                     ┆ f64           │
╞═════════════════════════╪═════════════════════╪═════════════════════════╪═══════════════╡
│ true                    ┆ 0.421623            ┆ 0.193174                ┆ 0.090029      │
│ false                   ┆ 0.650898            ┆ 0.06864                 ┆ 0.13138       │
└─────────────────────────┴─────────────────────┴─────────────────────────┴───────────────┘


### 3.3 Corrección importante: `factor_expansion` y `estrato_socioeconomico`

- **`factor_expansion`**: la ENDIREH 2021 define **dos** factores: **FAC_VIV** (vivienda) y **FAC_MUJ** (mujer elegida de 15+). Como cada fila de este dataset es una mujer entrevistada, `factor_expansion` casi con certeza corresponde a **FAC_MUJ**, no a FAC_VIV — usarlo así para ponderar las medidas de localización.
- **`estrato_socioeconomico`**: el descriptor oficial documenta ESTRATO/EST_DIS únicamente como campos de **diseño muestral geográfico** (ligados a tipo de localidad urbano/rural), **no** un índice de nivel de ingreso del hogar. Documentarla así en la tabla de clasificación de variables del Paso 5, y no interpretarla como proxy de riqueza.

## 4. Duplicados

In [14]:
n_filas_antes = df.height
n_duplicados = df.is_duplicated().sum()
print(f"Filas totales: {n_filas_antes}")
print(f"Filas duplicadas (exactas): {n_duplicados}")

df = df.unique()
n_filas_despues = df.height
print(f"Filas después de eliminar duplicados: {n_filas_despues}")
print(f"Filas eliminadas: {n_filas_antes - n_filas_despues}")

Filas totales: 110127
Filas duplicadas (exactas): 7882
Filas después de eliminar duplicados: 105753
Filas eliminadas: 4374


**Nota:** `df.unique()` detecta duplicados considerando *todas* las columnas. Si dos registros son iguales en todas las columnas excepto, por ejemplo, `cve_municipio` vs `nom_municipio` (redundantes entre sí), no se marcarán como duplicados y está bien — no son filas idénticas.

## 5. Códigos especiales → nulos reales

Confirmado contra el descriptor oficial (P13_14 y P4_5_AB):
- `edad_primer_union` = 98/99 → **98 = "No recuerda"**, **99 = "No especificado"**.
- `ingreso_pareja` = 999998/999999 → **999998 = "No sabe"**, **999999 = "No especificado"**.

**Detalle importante:** en `ingreso_pareja` (P4_5_AB), el código **999997** significa *"igual o más de $999,997"* — es **top-coding** (censura por confidencialidad), **no** un código de no-respuesta. No debe convertirse a nulo.

In [15]:
count_999997 = df.filter(pl.col("ingreso_pareja") == 999997).height
print(f"Registros con ingreso_pareja = 999997 (top-coded, NO es no-respuesta): {count_999997}")

Registros con ingreso_pareja = 999997 (top-coded, NO es no-respuesta): 63


In [16]:
df = df.with_columns([
    pl.when(pl.col("edad_primer_union").is_in([98, 99]))
      .then(None)
      .otherwise(pl.col("edad_primer_union"))
      .alias("edad_primer_union"),
    pl.when(pl.col("ingreso_pareja").is_in([999998, 999999]))
      .then(None)
      .otherwise(pl.col("ingreso_pareja"))
      .alias("ingreso_pareja"),
])

# Verificación: ya no deben aparecer esos códigos
print(df.filter(pl.col("edad_primer_union").is_in([98, 99])).height)  # debe ser 0
print(df.filter(pl.col("ingreso_pareja").is_in([999998, 999999])).height)  # debe ser 0

0
0


## 6. Imputación

In [17]:
total_filas = df.height
for col in ["edad_primer_union", "num_hijos", "ingreso_pareja"]:
    n_nulos = df[col].null_count()
    print(f"{col}: {n_nulos} nulos ({n_nulos / total_filas * 100:.1f}%)")

edad_primer_union: 82091 nulos (77.6%)
num_hijos: 19231 nulos (18.2%)
ingreso_pareja: 60860 nulos (57.5%)


### 6.1 `edad_primer_union`

Nulo estructural en mujeres que nunca han tenido pareja (confirmado por P3_8/P13_14). Imputar con la mediana/media sería conceptualmente incorrecto: asignaría una "edad de primera unión" a mujeres que nunca se han unido. En vez de imputar con un número, se agrega una variable indicadora.

In [ ]:
df = df.with_columns(
    pl.col("edad_primer_union").is_null().alias("nunca_tuvo_union")
)

# Para los cálculos de localización/variabilidad,
# se sigue trabajando solo con las mujeres que sí tienen el dato,
# tal como ya hace limpiar_cuantitativa() en medidas_localizacion.ipynb
# No se imputa esta variable con un valor numérico
#por lo explicado arriba
print(df["nunca_tuvo_union"].value_counts())

### 6.2 `num_hijos`

No hay un mapeo 1 a 1 confirmado con una única pregunta oficial, pero el faltante es más que sea no-respuesta real que "no aplica" estructural. Se imputa con la **mediana** por `estado_civil_desc` (más robusta que la media).

In [ ]:
mediana_num_hijos = df["num_hijos"].median()
print(f"Mediana global de num_hijos: {mediana_num_hijos}")

df = df.with_columns(
    pl.col("num_hijos").fill_null(
        pl.col("num_hijos").median().over("estado_civil_desc")
    )
)
print(f"Nulos restantes en num_hijos: {df['num_hijos'].null_count()}")

### 6.3 `ingreso_pareja`

Antes de decidir cómo tratar los nulos, se verifica si se explican por `estado_civil_desc` (sin pareja) o `pareja_trabaja_desc` (pareja no trabaja) — filtro oficial P4_3 → P4_5_AB del dataset

In [18]:
total_nulos_ingreso = df.filter(pl.col("ingreso_pareja").is_null()).height
print(f"Total de nulos en ingreso_pareja: {total_nulos_ingreso}")

nulos_por_estado_civil = (
    df.filter(pl.col("ingreso_pareja").is_null())
    .group_by("estado_civil_desc")
    .len()
    .sort("len", descending=True)
    .with_columns(
        (pl.col("len") / total_nulos_ingreso * 100).round(1).alias("% de los nulos")
    )
)
print(nulos_por_estado_civil)

Total de nulos en ingreso_pareja: 60860
shape: (6, 3)
┌───────────────────┬───────┬────────────────┐
│ estado_civil_desc ┆ len   ┆ % de los nulos │
│ ---               ┆ ---   ┆ ---            │
│ str               ┆ u32   ┆ f64            │
╞═══════════════════╪═══════╪════════════════╡
│ Casada            ┆ 26463 ┆ 43.5           │
│ Unión libre       ┆ 12187 ┆ 20.0           │
│ Soltera           ┆ 10634 ┆ 17.5           │
│ Viuda             ┆ 7104  ┆ 11.7           │
│ Separada          ┆ 3227  ┆ 5.3            │
│ Divorciada        ┆ 1245  ┆ 2.0            │
└───────────────────┴───────┴────────────────┘


In [19]:
nulos_por_pareja_trabaja = (
    df.filter(pl.col("ingreso_pareja").is_null())
    .group_by("pareja_trabaja_desc")
    .len()
    .sort("len", descending=True)
    .with_columns(
        (pl.col("len") / total_nulos_ingreso * 100).round(1).alias("% de los nulos")
    )
)
print(nulos_por_pareja_trabaja)

shape: (2, 3)
┌─────────────────────┬───────┬────────────────┐
│ pareja_trabaja_desc ┆ len   ┆ % de los nulos │
│ ---                 ┆ ---   ┆ ---            │
│ str                 ┆ u32   ┆ f64            │
╞═════════════════════╪═══════╪════════════════╡
│ No                  ┆ 56894 ┆ 93.5           │
│ Sí                  ┆ 3966  ┆ 6.5            │
└─────────────────────┴───────┴────────────────┘


In [20]:
cruce_nulos = (
    df.filter(pl.col("ingreso_pareja").is_null())
    .group_by(["estado_civil_desc", "pareja_trabaja_desc"])
    .len()
    .sort("len", descending=True)
)
print(cruce_nulos)

# ¿Cuántos nulos quedan en mujeres CON pareja y cuya pareja SÍ trabaja?
# Esos son los que de verdad parecen "no reportado" y no "no aplica".
nulos_sin_explicacion = df.filter(
    pl.col("ingreso_pareja").is_null()
    & (pl.col("estado_civil_desc") != "Soltera")
    & (pl.col("pareja_trabaja_desc") == "Sí")
).height
print(f"Nulos sin explicación estructural aparente: {nulos_sin_explicacion}")

shape: (12, 3)
┌───────────────────┬─────────────────────┬───────┐
│ estado_civil_desc ┆ pareja_trabaja_desc ┆ len   │
│ ---               ┆ ---                 ┆ ---   │
│ str               ┆ str                 ┆ u32   │
╞═══════════════════╪═════════════════════╪═══════╡
│ Casada            ┆ No                  ┆ 24928 │
│ Unión libre       ┆ No                  ┆ 11585 │
│ Soltera           ┆ No                  ┆ 9726  │
│ Viuda             ┆ No                  ┆ 6862  │
│ Separada          ┆ No                  ┆ 2797  │
│ …                 ┆ …                   ┆ …     │
│ Soltera           ┆ Sí                  ┆ 908   │
│ Unión libre       ┆ Sí                  ┆ 602   │
│ Separada          ┆ Sí                  ┆ 430   │
│ Divorciada        ┆ Sí                  ┆ 249   │
│ Viuda             ┆ Sí                  ┆ 242   │
└───────────────────┴─────────────────────┴───────┘
Nulos sin explicación estructural aparente: 3058


In [ ]:
nulos_por_estrato = (
    df.filter(pl.col("ingreso_pareja").is_null())
    .group_by("estrato_socioeconomico")
    .len()
    .sort("len", descending=True)
)
print(nulos_por_estrato)

**Decisión:** si la mayoría de los nulos cae en `estado_civil_desc == "Soltera"` o `pareja_trabaja_desc == "No"`, esos nulos tienen una razón de ser y **se dejan como nulo** (no se imputan). Solo se imputa con la mediana el remanente de mujeres con pareja que sí trabaja pero no reportó el ingreso (no-respuesta real, sin explicación estructural).

In [ ]:
mediana_ingreso = (
    df.filter(
        (pl.col("estado_civil_desc") != "Soltera")
        & (pl.col("pareja_trabaja_desc") == "Sí")
        & pl.col("ingreso_pareja").is_not_null()
    )["ingreso_pareja"].median()
)
print(f"Mediana de ingreso_pareja (con pareja que trabaja, dato reportado): {mediana_ingreso}")

df = df.with_columns(
    pl.when(
        pl.col("ingreso_pareja").is_null()
        & (pl.col("estado_civil_desc") != "Soltera")
        & (pl.col("pareja_trabaja_desc") == "Sí")
    )
    .then(mediana_ingreso)
    .otherwise(pl.col("ingreso_pareja"))  # todos los demás nulos se quedan como nulo
    .alias("ingreso_pareja")
)

In [ ]:
for col in ["edad_primer_union", "num_hijos", "ingreso_pareja"]:
    n_nulos = df[col].null_count()
    print(f"{col}: {n_nulos} nulos restantes")

## 8. Outliers: ¿se manejan o no?

En las instrucciones no se  exige tratar outliers explícitamente, pero es buena práctica detectarlos y documentar la decisión.

In [21]:
def detectar_outliers_iqr(df, columna):
    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers = df.filter(
        (pl.col(columna) < limite_inferior) | (pl.col(columna) > limite_superior)
    )
    print(f"{columna}: límites [{limite_inferior:.2f}, {limite_superior:.2f}]")
    print(f"  Outliers detectados: {outliers.height} de {df.height} ({outliers.height / df.height * 100:.2f}%)")
    return outliers

for col in ["edad_primer_union", "num_hijos", "ingreso_pareja"]:
    detectar_outliers_iqr(df, col)

edad_primer_union: límites [-17.50, 34.50]
  Outliers detectados: 990 de 105753 (0.94%)
num_hijos: límites [-2.00, 6.00]
  Outliers detectados: 0 de 105753 (0.00%)
ingreso_pareja: límites [-4500.00, 9900.00]
  Outliers detectados: 4921 de 105753 (4.65%)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ["edad_primer_union", "num_hijos", "ingreso_pareja"]):
    sns.boxplot(y=df[col].to_list(), ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Criterio por variable:**

| Variable | ¿Errores o valores reales? | ¿Se tratan? |
|---|---|---|
| `edad_primer_union` | Valores muy bajos (10-12 años) son extremos pero reales y son justo relevantes para el tema de estudio | No eliminar |
| `num_hijos` | Valores altos (8+) son posibles y reales | No eliminar, salvo un valor absurdo (20+) que sí sería error de captura |
| `ingreso_pareja` | Outliers extremos pueden mezclar valores reales con errores de captura o de escala. | Revisar caso por caso; usar la mediana (robusta) en las medidas descriptivas. |

**Decisión:** no se eliminan los outliers detectados, porque representan experiencias reales de la población encuestada (p. ej. uniones a edad muy temprana) y su eliminación sesgaría el análisis sobre un fenómeno social sensible

## 10. Conversión de variables Sí/No a binario

Las columnas categóricas de dos niveles `pareja_trabaja_desc`, `dinero_propio_desc`, `apoyo_gobierno_desc`, `tiene_ahorros_desc` y `propietaria_vivienda_desc` se almacenan como texto (`"Sí"` / `"No"`), a pesar de codificar en realidad una respuesta binaria. Se convierten a tipo entero, representando **`"Sí"` con 1 y `"No"` con 0**, siguiendo la misma convención que ya usa `sufrio_violencia_pareja` en el dataset original. Este cambio de tipo de dato (de `String` a `Int8`) facilita usarlas directamente en cálculos numéricos —promedios, proporciones, correlaciones— sin tener que comparar cadenas de texto en cada consulta.

El dataset también trae, para esas mismas 5 preguntas, una columna `_id` paralela codificada como **1/2** (`pareja_trabaja_id`, `dinero_propio_id`, `apoyo_gobierno_id`, `tiene_ahorros_id`, `propietaria_vivienda_id`). Antes de tocarlas se verificó, cruzando cada `_id` contra su `_desc` correspondiente, que la correspondencia es exacta y sin excepciones en las 5 variables: **1 siempre coincide con "Sí" y 2 siempre coincide con "No"**. Con esa confirmación, se recodifican también a **1/0** (1 se queda igual, 2 pasa a 0), para que ambas versiones de cada pregunta (`_id` y `_desc`) usen la misma convención binaria y sean intercambiables en cualquier cálculo posterior.

In [ ]:
pares_id_desc = [
    ("pareja_trabaja_id", "pareja_trabaja_desc"),
    ("dinero_propio_id", "dinero_propio_desc"),
    ("apoyo_gobierno_id", "apoyo_gobierno_desc"),
    ("tiene_ahorros_id", "tiene_ahorros_desc"),
    ("propietaria_vivienda_id", "propietaria_vivienda_desc"),
]

for id_col, desc_col in pares_id_desc:
    cruce = df.group_by([id_col, desc_col]).len().sort(id_col)
    print(f"--- {id_col} vs {desc_col} ---")
    print(cruce)

In [ ]:
columnas_si_no_desc = [
    "pareja_trabaja_desc", "dinero_propio_desc", "apoyo_gobierno_desc",
    "tiene_ahorros_desc", "propietaria_vivienda_desc",
]

df = df.with_columns([
    pl.col(col).replace({"Sí": 1, "No": 0}).cast(pl.Int8).alias(col)
    for col in columnas_si_no_desc
])

# Confirmado arriba: 1 = Sí, 2 = No en las 5 columnas _id.
# Se recodifican a 1/0 para que coincidan con la convención de las _desc.
columnas_si_no_id = [
    "pareja_trabaja_id", "dinero_propio_id", "apoyo_gobierno_id",
    "tiene_ahorros_id", "propietaria_vivienda_id",
]

df = df.with_columns([
    pl.col(col).replace({1: 1, 2: 0}).cast(pl.Int8).alias(col)
    for col in columnas_si_no_id
])

for col in columnas_si_no_desc + columnas_si_no_id:
    print(f"{col}: {df.schema[col]}")
    print(df[col].value_counts().sort(col))

## 11. Guardar el dataset limpio

In [ ]:
from config.rutas import RUTA_DATA_PROCESSED

ruta_salida = RUTA_DATA_PROCESSED / "endireh_2021_clean.csv"
df.write_csv(ruta_salida)
print(f"Dataset limpio guardado en: {ruta_salida}")
print(df.shape)